<a href="https://colab.research.google.com/github/Amazeble/chatterbox-finetuning/blob/main/colab_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook guides you through fine-tuning the Chatterbox TTS model.

## Step 1: Clone Repository & Install Dependencies

In [1]:
import os

# Clone the repository if not already present
if not os.path.exists("chatterbox-finetuning"):
    !git clone https://github.com/amazeble/chatterbox-finetuning.git
    %cd chatterbox-finetuning
else:
    %cd chatterbox-finetuning

from google.colab import drive
drive.mount('/content/drive')

Cloning into 'chatterbox-finetuning'...
remote: Enumerating objects: 872, done.
remote: Counting objects: 100% (211/211), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 872 (delta 139), reused 131 (delta 94), pack-reused 661 (from 1)
Receiving objects: 100% (872/872), 1.31 MiB | 4.28 MiB/s, done.
Resolving deltas: 100% (457/457), done.
/content/chatterbox-finetuning
Mounted at /content/drive


In [4]:
# Install FFmpeg
!apt-get update -qq
!apt-get install -y ffmpeg

# Install Python dependencies
!pip install chatterbox-tts --no-deps
!pip install torchcodec --index-url https://download.pytorch.org/whl/cu124 --force-reinstall
!pip install peft==0.17.1  --no-deps
!pip install silero-vad==6.2.0 librosa==0.11.0 soundfile==0.13.1 num2words ffmpeg-python tqdm pandas safetensors tensorboard omegaconf hf_transfer pyloudnorm gdown




W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 145 not upgraded.
Looking in indexes: https://download.pytorch.org/whl/cu124
  Using cached https://download.pytorch.org/whl/cu124/TorchCodec-0.2.1%2Bcu124-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
Using cached https://download.pytorch.org/whl/cu124/TorchCodec-0.2.1%2Bcu124-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (846 kB)
  Attempting uninstall: torchcodec
    Found existing installation: TorchCodec 0.2.1+cu124
    Uninstalling TorchCodec-0.2.1+cu124:
      Successfully uninstalled TorchCodec-0.2.1+cu124
  Preparing metadata (setup.py) ... don

## Step 2: Configure Training Settings

In [ ]:
#@title ⚙️ STEP 0: Configuration
#@markdown ### 📂 Project Settings
project_name = "Adriene" #@param {type:"string"}

#@markdown ### 📁 Paths
model_dir = "./pretrained_models" #@param {type:"string"}
base_dataset_dir = "./MyTTSDataset" #@param {type:"string"}
base_output_dir = "./chatterbox_output" #@param {type:"string"}

#@markdown ### 🗃️ Dataset Format
preprocess = True #@param ["True", "False", "auto"]
preprocess = preprocess == "True" or (preprocess == "auto" and True)  # Convert to boolean

#@markdown ### 🏋️ Training Mode
is_turbo = True #@param ["True", "False"]
is_turbo = is_turbo == "True"
is_lora = True #@param ["True", "False"]
is_lora = is_lora == "True"
is_merge_lora = False #@param ["True", "False"]
is_merge_lora = is_merge_lora == "True"

#@markdown ### 📊 Hyperparameters
batch_size = 8 #@param ["2", "4", "8", "16"]
batch_size = int(batch_size)
grad_accum = 4 #@param ["1", "2", "4", "8"]
grad_accum = int(grad_accum)
learning_rate = 0.0001 #@param {type:"number"}
num_epochs = 10 #@param {type:"integer"}
save_steps = 500 #@param {type:"integer"}
save_total_limit = 5 #@param {type:"integer"}
dataloader_num_workers = 8 #@param {type:"integer"}

# Save config override for train.py to read
import json
config_override = {
    'project_name': project_name,
    'is_turbo': is_turbo,
    'is_lora': is_lora,
    'is_merge_lora': is_merge_lora,
    'batch_size': batch_size,
    'grad_accum': grad_accum,
    'learning_rate': learning_rate,
    'num_epochs': num_epochs,
    'save_steps': save_steps,
    'save_total_limit': save_total_limit,
    'dataloader_num_workers': dataloader_num_workers
}
with open('colab_config_override.json', 'w') as f:
    json.dump(config_override, f)

# Directly modify src/config.py to hardcode paths
csv_path = f"{base_dataset_dir}/{project_name}/metadata.csv" if project_name else f"{base_dataset_dir}/metadata.csv"
wav_dir = f"{base_dataset_dir}/{project_name}/wavs" if project_name else f"{base_dataset_dir}/wavs"
preprocessed_dir = f"{base_dataset_dir}/{project_name}/preprocess" if project_name else f"{base_dataset_dir}/preprocess"
output_dir = f"{base_output_dir}/{project_name}" if project_name else base_output_dir

with open('src/config.py', 'r') as f:
    config_content = f.read()

# Replace project_name default
config_content = config_content.replace(
    'project_name: str = field(default_factory=lambda: _require_field_value("project_name"))',
    f'project_name: str = "{project_name}"'
)

# Replace path defaults
config_content = config_content.replace(
    'csv_path: str = None  # Set dynamically in __post_init__',
    f'csv_path: str = "{csv_path}"'
)
config_content = config_content.replace(
    'wav_dir: str = None  # Set dynamically in __post_init__',
    f'wav_dir: str = "{wav_dir}"'
)
config_content = config_content.replace(
    'preprocessed_dir: str = None  # Set dynamically in __post_init__',
    f'preprocessed_dir: str = "{preprocessed_dir}"'
)
config_content = config_content.replace(
    'output_dir: str = None  # Set dynamically in __post_init__',
    f'output_dir: str = "{output_dir}"'
)

# Comment out __post_init__ path resolution since paths are now hardcoded
if 'def __post_init__(self):' in config_content:
    lines = config_content.split('\n')
    new_lines = []
    in_post_init = False
    for line in lines:
        if 'def __post_init__(self):' in line:
            in_post_init = True
            new_lines.append('# '+ line)
        elif in_post_init and (line.startswith('        ') or line.startswith('\t')):
            new_lines.append('# '+ line)
        else:
            in_post_init = False
            new_lines.append(line)
    config_content = '\n'.join(new_lines)

with open('src/config.py', 'w') as f:
    f.write(config_content)

print(f'✅ Config saved: project_name="{project_name}"')
print(f'   Paths: csv={csv_path}, wav={wav_dir}')
print(f'   Training: turbo={is_turbo}, lora={is_lora}, merge={is_merge_lora}')
print(f'   Hyperparams: batch={batch_size}, grad_accum={grad_accum}, lr={learning_rate}, epochs={num_epochs}')


In [ ]:
%cd chatterbox-finetuning/

## Step 3: Download and Prepare Base Models

In [5]:
# Clean pretrained_models if switching modes
if os.path.exists("pretrained_models"):
    import shutil
    shutil.rmtree("pretrained_models")
    print("🗑️ Cleaned old pretrained_models directory")

# Run setup.py
print("\n⏳ Downloading and preparing models...\n")
!python setup.py

print("\n✅ Setup complete! Check the output above for the new_vocab_size value.")


⏳ Downloading and preparing models...

--- Chatterbox Pretrained Model Setup ---

Creating directory: pretrained_models
Mode: CHATTERBOX-TURBO (Checking 10 files)
Downloading: ve.safetensors...
ve.safetensors: 100% 5.43M/5.43M [00:00<00:00, 20.1MiB/s]
Download complete: pretrained_models/ve.safetensors

Downloading: t3_turbo_v1.safetensors...
t3_turbo_v1.safetensors: 100% 1.78G/1.78G [00:23<00:00, 81.8MiB/s]
Download complete: pretrained_models/t3_turbo_v1.safetensors

Downloading: s3gen_meanflow.safetensors...
s3gen_meanflow.safetensors: 100% 0.99G/0.99G [00:13<00:00, 81.8MiB/s]
Download complete: pretrained_models/s3gen_meanflow.safetensors

Downloading: conds.pt...
conds.pt: 100% 165k/165k [00:00<00:00, 755kiB/s]  
Download complete: pretrained_models/conds.pt

Downloading: vocab.json...
vocab.json: 976kiB [00:00, 41.8MiB/s]
Download complete: pretrained_models/vocab.json

Downloading: added_tokens.json...
added_tokens.json: 100% 418/418 [00:00<00:00, 3.18MiB/s]
Download complete: 

## Step 4: Start Training

In [ ]:
!python train.py --project_name "$project_name"

In [9]:
pip install resemble-perth


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.4/34.4 MB 67.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chatterbox-tts 0.1.7 requires conformer==0.3.2, which is not installed.
chatterbox-tts 0.1.7 requires pykakasi==2.3.0, which is not installed.
chatterbox-tts 0.1.7 requires s3tokenizer, which is not installed.
chatterbox-tts 0.1.7 requires spacy-pkuseg, which is not installed.
chatterbox-tts 0.1.7 requires diffusers==0.29.0, but you have diffusers 0.39.0 which is incompatible.
chatterbox-tts 0.1.7 requires gradio==6.8.0, but you have gradio 6.20.0 which is incompatible.
chatterbox-tts 0.1.7 requires numpy<2.0.0,>=1.24.0; python_version < "3.13", but you have numpy 2.0.2 which is incompatible.
chatterbox-tts 0.1.7 requires safetensors==0.5.3, but you have safetensors 0.8.0 which is incompatible.
chatterbox-tts 0.1.7 requires torch==2.6.0

In [ ]:
pip install chatterbox-tts==0.1.7

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 113.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 107.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/8

In [7]:
pip install "transformers<5.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 116.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.23.0
    Uninstalling huggingface_hub-1.23.0:
      Successfully uninstalled huggingface_hub-1.23.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chatterbox-tts 0.1.7 requires conformer==0.3.2, which is not installed.
chatterbox-tts 0.1.7 requires pykakasi==2.3.0, which is not installed.
chatterbox-tts 0.1.7 requires resemble-perth>=1.0.0, which is not insta

##Test Inference

In [ ]:
#@title 🗣️ Test Speech Synthesis (Inference)
#@markdown Generate speech using your trained model. Make sure you have a reference audio file.

import os
from IPython.display import Audio, display

# Check for speaker reference
ref_dir = "speaker_reference"
if not os.path.exists(ref_dir):
    os.makedirs(ref_dir)
    print(f"Created {ref_dir}/ directory")
    print("\n⚠️ Please upload a reference audio file (3-10 seconds clean speech)")
    print(f"   Upload it to: {ref_dir}/reference.wav")
else:
    ref_files = [f for f in os.listdir(ref_dir) if f.endswith('.wav')]
    if ref_files:
        print(f"Found reference files: {ref_files}")

        # Update inference.py with test text
        test_text = "Hello, this is a test of my custom voice model."  #@param {type:"string"}
        ref_file = ref_files[0]  #@param {type:"string"}

        # Modify inference.py temporarily
        with open('inference.py', 'r') as f:
            inf_content = f.read()

        # Replace test text
        inf_content = inf_content.replace(
            'TEXT_TO_SAY = "Merhaba, sesimi geliştirmem oldukça uzun zaman aldı ve şimdi sahip olduğuma göre, sessiz kalmayacağım."',
            f'TEXT_TO_SAY = "{test_text}"'
        )

        with open('inference.py', 'w') as f:
            f.write(inf_content)

        print(f"\nRunning inference with:")
        print(f"  Text: {test_text}")
        print(f"  Reference: {ref_dir}/{ref_file}")
        print("\nGenerating speech...\n")

        # Run inference
        !python inference.py

        # Play output
        if os.path.exists("output.wav"):
            print("\n✅ Generated output.wav")
            display(Audio("output.wav"))
        else:
            print("\n❌ Output file not generated. Check error messages above.")
    else:
        print(f"\n⚠️ No .wav files found in {ref_dir}/")
        print("Please upload a reference audio file (3-10 seconds of clean speech)")

##Merge LoRA Adapter (Optional)

In [ ]:
#@title 📦 Merge LoRA Adapter into Base Model
#@markdown If you used LoRA training and are satisfied with the results, merge the adapter into a standalone model file.

if is_lora:
    print("Merging LoRA adapter into base model...")
    print("This creates a single .safetensors file ready for deployment.\n")

    !python merge_lora.py

    print("\n✅ Merge complete!")
    merged_model = f"chatterbox_output/{project_name}/t3_turbo_finetuned_merged.safetensors" if is_turbo else f"chatterbox_output/{project_name}/t3_finetuned_merged.safetensors"
    print(f"Merged model saved to: {merged_model}")

    if os.path.exists(merged_model):
        size_mb = os.path.getsize(merged_model) / (1024 * 1024)
        print(f"File size: {size_mb:.1f} MB")
else:
    print("ℹ️ Skipping merge - you used full fine-tuning, not LoRA")

##Download Trained Model

In [ ]:
#@title 💾 Download Trained Model
#@markdown Download your trained model to Google Drive or local storage.

from google.colab import drive
import os
import shutil

# Mount Google Drive
drive.mount('/content/drive', force_remount=False)

# Determine what to save
# Use config to get the correct output_dir
from src.config import TrainConfig
cfg = TrainConfig()
output_dir = cfg.output_dir

if os.path.exists(output_dir):
    # Create backup directory in Drive
    drive_backup = f"/content/drive/MyDrive/chatterbox_models/{project_name}"
    os.makedirs(drive_backup, exist_ok=True)

    # Copy all output files
    for item in os.listdir(output_dir):
        src = os.path.join(output_dir, item)
        dst = os.path.join(drive_backup, item)

        if os.path.isfile(src):
            shutil.copy2(src, dst)
            print(f"✅ Copied: {item} ({os.path.getsize(src) / (1024*1024):.1f} MB)")
        elif os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
            print(f"✅ Copied directory: {item}/")

    print(f"\n📁 All files saved to: {drive_backup}")
    print("\nYou can also download individual files using the file browser on the left.")
else:
    print(f"❌ Output directory not found: {output_dir}")
    print("Make sure training has completed successfully.")

---
## Troubleshooting Tips

### Common Issues:

1. **CUDA Out of Memory**
   - Enable LoRA training (`is_lora = True`)
   - Reduce `batch_size` in `src/config.py`
   - Use a smaller dataset

2. **Poor Quality Output**
   - Ensure reference audio is clean (3-10 seconds)
   - Check dataset quality (minimal background noise)
   - Train for more epochs if dataset is small
   - For Turbo model: switch to LoRA if experiencing instability

3. **Vocabulary Mismatch Error**
   - Make sure `new_vocab_size` matches the tokenizer
   - Re-run `setup.py` after changing modes

4. **Missing Reference Audio**
   - Upload a clean 3-10 second WAV file to `speaker_reference/`
   - Use the same speaker as your training dataset for best results

---
**Need Help?** Check the README.md file or open an issue on GitHub.